In [16]:
import os
import numpy as np
import pandas as pd
import lightkurve as lk
from lightkurve import LightCurveCollection
from tqdm import tqdm
from sklearn.utils import shuffle, resample
from typing import List, Optional
from scipy.signal import savgol_filter
from requests.exceptions import ReadTimeout, ConnectionError, ChunkedEncodingError
import time

In [17]:
all_df = pd.read_csv("../data/all_df.csv")
all_df['catalog'].value_counts()

C:\Users\yashs\AppData\Local\Temp\ipykernel_4296\3335589737.py:1: DtypeWarning: Columns (2,3,6,9,26,27,28,29,30,36,41,105,131,132,134,135,136,137,141,152,159,163,169,173,241,242,243) have mixed types. Specify dtype option on import or set low_memory=False.
  all_df = pd.read_csv("../data/all_df.csv")


catalog
TCE    34032
KOI     9564
TOI     7668
Name: count, dtype: int64

In [31]:
def get_local_view(folded_lc, duration, n_bins=201):
    """
    Creates a 'local view' of a transit.

    This function takes a folded light curve, isolates the region around the
    transit (phase 0), and bins it to a fixed size.

    Args:
        folded_lc (lightkurve.FoldedLightCurve): The light curve, folded on the period.
        duration (float): The transit duration in the same units as the light curve's time.
        n_bins (int): The number of bins for the output view. Must be odd.

    Returns:
        np.ndarray: A binned, normalized 1D array representing the local view.
    """
    if n_bins % 2 == 0:
        raise ValueError("n_bins must be an odd number.")

    # Isolate the region around the transit (e.g., twice the duration)
    phase_mask = (folded_lc.phase.value > -duration) & (folded_lc.phase.value < duration)
    lc_zoom = folded_lc[phase_mask]

    # Bin the zoomed-in light curve to a fixed size
    binned_lc = lc_zoom.bin(time_bin_size= (2 * duration) / n_bins)
    
    # Normalize the flux
    normalized_flux = binned_lc.flux.value / np.nanmedian(binned_lc.flux.value)
    
    # Ensure the output has the exact length, padding if necessary
    final_flux = np.full(n_bins, np.nan)
    start_index = (n_bins - len(normalized_flux)) // 2
    
    if start_index >= 0 and start_index + len(normalized_flux) <= n_bins:
      final_flux[start_index : start_index + len(normalized_flux)] = normalized_flux
    
    # Fill any remaining NaNs with 1.0 (representing no transit)
    final_flux[np.isnan(final_flux)] = 1.0
    
    return final_flux


def get_global_view(folded_lc, period, n_bins=2001):
    """
    Creates a 'global view' of the entire orbital phase.

    This function bins the entire folded light curve to show the full orbit,
    helping the model distinguish a transit from other stellar variability.

    Args:
        folded_lc (lightkurve.FoldedLightCurve): The light curve, folded on the period.
        period (float): The orbital period.
        n_bins (int): The number of bins for the output view.

    Returns:
        np.ndarray: A binned, normalized 1D array of the global view.
    """
    # Bin the entire folded light curve
    binned_lc = folded_lc.bin(time_bin_size=period / n_bins)
    
    # Normalize the flux
    normalized_flux = binned_lc.flux.value / np.nanmedian(binned_lc.flux.value)
    
    # Pad or truncate to ensure it has the exact length
    final_flux = np.full(n_bins, np.nan)
    if len(normalized_flux) >= n_bins:
        final_flux = normalized_flux[:n_bins]
    else:
        final_flux[:len(normalized_flux)] = normalized_flux

    # Fill any remaining NaNs with 1.0
    final_flux[np.isnan(final_flux)] = 1.0

    return final_flux

In [18]:
def safe_download(target_id: str, mission: str, author: str, fast_mode: bool = False):
    """
    Robustly downloads light curve data, handling network errors, missing data,
    and a 'fast_mode' to download only a subset of data.

    Args:
        target_id (str): The star's ID (e.g., "KIC 1234567").
        mission (str): The mission ('Kepler', 'TESS', etc.).
        author (str): The data author (e.g., 'Kepler', 'SPOC').
        fast_mode (bool): If True, downloads only the first found light curve file.
                          If False, downloads all available files.

    Returns:
        lightkurve.LightCurve: A cleaned LightCurve object, or None if any step fails.
    """
    retries = 3
    wait_time = 5  # seconds

    while retries > 0:
        try:
            search_result = lk.search_lightcurve(target_id, mission=mission, author=author)

            # 1. Check if the star exists in the archive
            if not search_result:
                print(f"[Warning] No search results for {target_id}. Skipping.")
                return None

            # 2. Download data based on fast_mode
            if fast_mode:
                # Download only the first light curve file in the search result
                lc_collection = search_result[0].download()
            else:
                # Download all available light curves
                lc_collection = search_result.download_all()

            # 3. Check if download returned any data
            if not lc_collection:
                print(f"[Warning] Search found for {target_id}, but download returned nothing. Skipping.")
                return None

            # 4. Stitch and clean. This works on both single and multiple light curves.
            lc = lc_collection.stitch().remove_nans()

            # 5. Check if the final light curve has any data points left
            if len(lc) == 0:
                print(f"[Warning] Data for {target_id} is empty after cleaning. Skipping.")
                return None

            # If all checks pass, success!
            return lc

        except (ReadTimeout, ConnectionError, ChunkedEncodingError) as e:
            print(f"[Network Error] {e} for {target_id}. Retrying... ({retries-1} left)")
            time.sleep(wait_time)
            retries -= 1
        except Exception as e:
            print(f"[Error] An unexpected error occurred for {target_id}: {e}")
            return None # Exit immediately for other errors

    print(f"[Failure] All retries failed for {target_id}. Skipping.")
    return None

In [19]:
def balance_catalog(df, catalog, n_samples=None):
    df = df[df["catalog"] == catalog]
    df = df[df["period"].notnull() & df["epoch"].notnull()]
    
    positives = df[df["label"]==1]
    negatives = df[df["label"]==0]

    n = min(len(positives), len(negatives))
    if n_samples:
        n = min(n, n_samples)
    
    pos = positives.sample(n, random_state=42)
    neg = negatives.sample(n, random_state=42)
    
    return pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)

In [20]:
def process_lightcurve(lc, period, epoch, duration_hours):
    """
    Processes a light curve, correctly converting duration from hours to phase.
    """
    try:
        if lc is None or len(lc) == 0: return None, None
        # Check for valid duration as well
        if not all(np.isfinite([period, epoch, duration_hours])) or period <= 0 or duration_hours <= 0:
            return None, None
        if np.std(lc.flux.value) < 1e-6: return None, None
        
        # --- THE CRITICAL FIX ---
        # 1. Convert duration from hours to days
        duration_days = duration_hours / 24.0
        # 2. Convert duration from days to phase units
        duration_phase = duration_days / period
        
        folded_lc = lc.fold(period=period, epoch_time=epoch)
        
        # Pass the correctly scaled duration to get_local_view
        local_view = get_local_view(folded_lc, duration=duration_phase)
        global_view = get_global_view(folded_lc, period=period)
        
        return global_view, local_view
    except Exception:
        return None, None

In [21]:
def build_dataset_resumable(df, mission, id_prefix, output_dir="results", fast_mode=False):
    """
    Processes a dataframe of targets and saves each result as a separate .npz file.
    This allows the process to be stopped and resumed.
    """
    # Create the directory for our results if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Processing {len(df)} targets. Results will be saved in '{output_dir}/'")
    
    # Using tqdm for a nice progress bar
    for _, row in tqdm(df.iterrows(), total=len(df)):
        target_id = f"{id_prefix} {int(row['target_id'])}"
        output_filename = os.path.join(output_dir, f"{target_id.replace(' ', '_')}.npz")

        # --- THE RESUME LOGIC ---
        # If the output file already exists, skip this star
        if os.path.exists(output_filename):
            continue

        # Use the robust functions we developed
        # Note: I've simplified the author/mission logic for this example
        author = "Kepler" if mission == "Kepler" else "SPOC"
        lc = safe_download(target_id, mission, author, fast_mode=fast_mode)
        
        global_view, local_view = process_lightcurve(lc, row['period'], row['epoch'], duration=0.5)

        # If processing was successful, save the result for this single star
        if global_view is not None and local_view is not None:
            np.savez(
                output_filename,
                X_global=global_view,
                X_local=local_view,
                y=np.array([row['label']], dtype=np.int8) # Save the label as an array
            )


In [22]:
def consolidate_results(output_dir="results"):
    """
    Loads all individual .npz files from a directory and merges them
    into the final training arrays.
    """
    print(f"Consolidating results from '{output_dir}/'...")
    files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.npz')]
    
    Xg_list, Xl_list, y_list = [], [], []

    for f in tqdm(files):
        with np.load(f) as data:
            Xg_list.append(data['X_global'])
            Xl_list.append(data['X_local'])
            y_list.append(data['y'])
    
    # Stack the lists of arrays into final, single arrays
    Xg = np.vstack(Xg_list)
    Xl = np.vstack(Xl_list)
    y  = np.hstack(y_list)
    
    print(f"Finished consolidation. Final dataset shape: {Xg.shape}")
    return Xg, Xl, y

In [23]:
koi_balanced = balance_catalog(all_df, "KOI", n_samples=2000)
toi_balanced = balance_catalog(all_df, "TOI", n_samples=2000)

In [33]:
print("[INFO] Building KOI dataset...")
build_dataset_resumable(
    df=koi_balanced,
    mission="Kepler",
    id_prefix="KIC",
    output_dir="results_koi",
    fast_mode=False
)

[INFO] Building KOI dataset...
Processing 4000 targets. Results will be saved in 'results_koi/'


  0%|▏                                                                                | 8/4000 [00:01<16:30,  4.03it/s]

[Error] An unexpected error occurred for KIC 12068975: Not recognized as a supported data product:
D:\exoplanet_ai\notebooks\mastDownload\Kepler\kplr012068975_sc_Q000000033333333332\kplr012068975-2011073133259_slc.fits
This file may be corrupt due to an interrupted download. Please remove it from your disk and try again.


  1%|▋                                                                         | 40/4000 [1:01:33<101:33:45, 92.33s/it]


KeyboardInterrupt: 

In [ ]:
# --- 2. Call the same function for your TOI (TESS) data ---
print("\n[INFO] Building TOI dataset...")
build_dataset_resumable(
    df=toi_balanced,
    mission="TESS",
    id_prefix="TIC",
    output_dir="results_toi",
    fast_mode=False
)